# Teste avaliativo para vaga de bolsista em engenharia/análise de dados

In [51]:
import requests
import time
import json
import pandas as pd
import pickle
import os

In [58]:
# CONFIGURAR PASTA PARA SALVAR OS DADOS DA REQUISIÇÃO
os.makedirs("data", exist_ok=True)


URL = 'https://api.obrasgov.gestao.gov.br/obrasgov/api'
ENDPOINT = 'projeto-investimento'
MAX_ATTEMPT = 2
# MAX_PAGE = 42 # a partir da página 43 na data 15/10/2025, não existe mais dado (= 20 TAMANHO PAGINA)
PAGE_SIZE = 10
UF = 'DF'
page = 0 # pagina atual


# apenas projetos que são do DF
projetos_DF = []


while True:
    response = requests.get(f'{URL}/{ENDPOINT}?uf={UF}&pagina={page}&tamanhoDaPagina={PAGE_SIZE}')

    # se vir um erro 404 indica que aquela página não existe ou a requisição está falhando
    if response.status_code == 404:
        print(f"Página {page} não existe. Finalizando.")
        break
    
    # se vir um erro 429, siginifica que meu ip tá sendo negado, preciso aguardar o rate-limit imposto pela api
    elif response.status_code == 429:
        retry_after = int(response.headers.get("x-rate-limit-retry-after-seconds", 2))
        print(f"Muitas requisições! Aguardando {retry_after} segundos...")
        time.sleep(retry_after)
        continue
    
    try:
        data = response.json()
    except ValueError as e:
        print(f"Erro ao decodificar JSON na página {page}: {e}")
        break
        
    if not data.get("content") or data.get("empty", False):
        print(f"Acabaram os dados para o UF {UF}! ({page=})")
        break
    
    # pegas apenas dados do DF
    for item in data.get("content", []):
        if item['uf'] == 'DF':
            projetos_DF.append(item)
    
    print(len(projetos_DF))
    
    page += 1
    time.sleep(2) # EVITAR TOMAR STATUS CODE 429

print('Terminou')

# SALVAR EM JSON:
json_path = os.path.join('data', 'projetos_DF.json')
with open(json_path, "w", encoding="utf-8") as file:
    json.dump(projetos_DF, file, ensure_ascii=False, indent=2)
print(f'Arquivo JSON salvo: {json_path}')


# SALVAR EM CSV:
csv_path = os.path.join('data', 'projetos_DF.csv')
df = pd.DataFrame(projetos_DF)
df.to_csv(csv_path, index=False, encoding="utf-8")
print(f'Arquivo CSV salvo: {csv_path}')


# SALVAR EM PICKLE:
pkl_path = os.path.join('data', 'projetos_DF.pkl')
with open(pkl_path, "wb") as file:
    pickle.dump(projetos_DF, file)
print(f'Arquivo Pickle salvo: {pkl_path}')

20
40
60
80
100
120
140
160
180
200
220
240
260
280
300
320
340
360
380
400
420
440
460
480
500
520
540
560
580


KeyboardInterrupt: 